# Defense Evaluation: Watermark, Validation, Proactive, Composite

This notebook evaluates four defense mechanisms against the three attack methodologies:

- **Watermark Defense**: Unigram z-score detection (Zhao et al., ICLR 2024, arXiv:2306.17439)
- **Content Validation Defense**: Pattern-based adversarial content filtering
- **Proactive Defense**: Pre-emptive attack simulation and vulnerability assessment
- **Composite Defense**: Multi-layer combination of all three defenses

**Defense Metric Definitions**:
- **TPR** (True Positive Rate): Fraction of adversarial entries correctly detected
- **FPR** (False Positive Rate): Fraction of benign entries incorrectly flagged
- **F1**: Harmonic mean of precision and recall
- **Utility**: Fraction of benign queries answered correctly despite defense

In [ ]:
import json
import os
import sys

import matplotlib.pyplot as plt
import numpy as np

sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'src'))

from attacks.implementations import AttackSuite
from data.synthetic_corpus import SyntheticCorpus
from evaluation.benchmarking import DefenseEvaluator
from evaluation.retrieval_sim import RetrievalSimulator

matplotlib.rcParams['figure.dpi'] = 150
matplotlib.rcParams['font.size'] = 11
matplotlib.rcParams['axes.spines.top'] = False
matplotlib.rcParams['axes.spines.right'] = False

SEED = 42
CORPUS_SIZE = 200
print('defense evaluation setup complete')

## 1. Generate Poisoned Content Samples

In [ ]:
corpus = SyntheticCorpus(seed=SEED)
benign_entries = corpus.generate_benign_entries(50)
clean_content = [e['content'] for e in benign_entries]

# generate poisoned content via attack suite
attack_suite = AttackSuite()
poisoned_content = []

for entry in benign_entries[:20]:
    results = attack_suite.execute_all(entry['content'])
    for at, result in results['attack_results'].items():
        pc = (
            result.get('poisoned_content')
            or result.get('injected_content')
            or result.get('manipulated_content')
        )
        if pc and isinstance(pc, str):
            poisoned_content.append(pc)

print(f'clean content samples: {len(clean_content)}')
print(f'poisoned content samples: {len(poisoned_content)}')
print(f'\nsample clean: {clean_content[0][:80]}')
print(f'sample poisoned: {poisoned_content[0][:80] if poisoned_content else "n/a"}')

## 2. Evaluate Each Defense

In [ ]:
defense_evaluator = DefenseEvaluator()
defense_types = ['watermark', 'validation', 'proactive', 'composite']

defense_results = {}
for dt in defense_types:
    m = defense_evaluator.evaluate_defense(
        dt, attack_suite, clean_content, poisoned_content
    )
    defense_results[dt] = m
    print(f'\n{dt}:')
    print(f'  TPR={m.tpr:.3f}  FPR={m.fpr:.3f}  Precision={m.precision:.3f}  F1={m.f1_score:.3f}')
    print(f'  TP={m.true_positives}  FP={m.false_positives}  TN={m.true_negatives}  FN={m.false_negatives}')

## 3. Defense Performance Bar Chart (TPR, FPR, F1)

In [ ]:
defense_labels = {
    'watermark': 'Watermark\n(Zhao et al.)',
    'validation': 'Content\nValidation',
    'proactive': 'Proactive\nDefense',
    'composite': 'Composite\n(Multi-layer)',
}

x = np.arange(len(defense_types))
width = 0.25
colors = {'tpr': '#27AE60', 'fpr': '#E74C3C', 'f1_score': '#3498DB'}

fig, ax = plt.subplots(figsize=(12, 6))

for i, (metric, color, label) in enumerate([
    ('tpr', '#27AE60', 'TPR (Detection Rate)'),
    ('fpr', '#E74C3C', 'FPR (False Positive Rate)'),
    ('f1_score', '#3498DB', 'F1 Score'),
]):
    values = [getattr(defense_results[dt], metric) for dt in defense_types]
    bars = ax.bar(x + (i - 1) * width, values, width, color=color, alpha=0.85,
                  edgecolor='black', linewidth=0.5, label=label)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
                f'{val:.2f}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels([defense_labels[dt] for dt in defense_types])
ax.set_ylabel('Score')
ax.set_title('Defense Performance Metrics: TPR, FPR, and F1 Score', fontsize=13)
ax.set_ylim(0, 1.15)
ax.legend(loc='upper right')
ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.savefig('../visualization/fig_defense_performance_comparison.png', dpi=300, bbox_inches='tight')
plt.savefig('../visualization/fig_defense_performance_comparison.pdf', bbox_inches='tight')
plt.show()
print('Figure saved: fig_defense_performance_comparison')

## 4. ROC Curve: Defense Trade-off Visualization

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))

defense_colors = {
    'watermark': '#8E44AD',
    'validation': '#E67E22',
    'proactive': '#16A085',
    'composite': '#C0392B',
}
defense_markers = {'watermark': 'o', 'validation': 's', 'proactive': '^', 'composite': 'D'}

# random classifier baseline
ax.plot([0, 1], [0, 1], 'k--', alpha=0.4, linewidth=1, label='Random classifier')
ax.fill_between([0, 1], [0, 1], [1, 1], alpha=0.05, color='green')
ax.text(0.05, 0.95, 'Ideal region', color='darkgreen', fontsize=9, alpha=0.7)

for dt in defense_types:
    m = defense_results[dt]
    ax.scatter(
        m.fpr, m.tpr,
        c=defense_colors[dt],
        marker=defense_markers[dt],
        s=180,
        zorder=5,
        label=f'{defense_labels[dt].replace(chr(10), " ")} (F1={m.f1_score:.2f})',
        edgecolors='black',
        linewidths=1,
    )

ax.set_xlabel('False Positive Rate (FPR)', fontsize=12)
ax.set_ylabel('True Positive Rate (TPR)', fontsize=12)
ax.set_title('Defense ROC Space: Effectiveness vs False Positive Trade-off', fontsize=12)
ax.set_xlim(-0.05, 1.05)
ax.set_ylim(-0.05, 1.05)
ax.legend(loc='lower right', fontsize=9)
ax.set_aspect('equal')

plt.tight_layout()
plt.savefig('../visualization/fig_defense_roc_space.png', dpi=300, bbox_inches='tight')
plt.savefig('../visualization/fig_defense_roc_space.pdf', bbox_inches='tight')
plt.show()
print('Figure saved: fig_defense_roc_space')

## 5. Attack-Defense Interaction Matrix

In [ ]:
# run each attack then evaluate each defense's effectiveness against those specific attack outputs
attack_types = ['agent_poison', 'minja', 'injecmem']
attack_labels = {'agent_poison': 'AgentPoison', 'minja': 'MINJA', 'injecmem': 'InjecMEM'}

interaction_matrix = np.zeros((len(defense_types), len(attack_types)))

for j, at in enumerate(attack_types):
    # generate poisoned content specific to this attack
    atk = attack_suite.get_attack(at)
    attack_poison = []
    for entry in benign_entries[:15]:
        result = atk.execute(entry['content'])
        pc = (
            result.get('poisoned_content')
            or result.get('injected_content')
            or result.get('manipulated_content')
        )
        if pc and isinstance(pc, str):
            attack_poison.append(pc)

    for i, dt in enumerate(defense_types):
        m = defense_evaluator.evaluate_defense(dt, attack_suite, clean_content[:15], attack_poison)
        # effectiveness = TPR - FPR (youden index)
        interaction_matrix[i, j] = m.tpr - m.fpr

fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(interaction_matrix, cmap='RdYlGn', vmin=-0.5, vmax=1.0, aspect='auto')
ax.set_xticks(range(len(attack_types)))
ax.set_xticklabels([attack_labels[at] for at in attack_types])
ax.set_yticks(range(len(defense_types)))
ax.set_yticklabels([defense_labels[dt].replace('\n', ' ') for dt in defense_types])
ax.set_title('Attack-Defense Interaction Matrix\n(Youden Index: TPR - FPR; Higher is Better Defense)', fontsize=11)
plt.colorbar(im, ax=ax, label='TPR - FPR (Youden Index)')

for i in range(len(defense_types)):
    for j in range(len(attack_types)):
        ax.text(j, i, f'{interaction_matrix[i, j]:.2f}', ha='center', va='center',
                fontsize=11, fontweight='bold',
                color='white' if abs(interaction_matrix[i, j]) > 0.5 else 'black')

plt.tight_layout()
plt.savefig('../visualization/fig_attack_defense_interaction_matrix.png', dpi=300, bbox_inches='tight')
plt.savefig('../visualization/fig_attack_defense_interaction_matrix.pdf', bbox_inches='tight')
plt.show()
print('Figure saved: fig_attack_defense_interaction_matrix')

## 6. Defense Impact on ASR-R (Retrieval Reduction)

In [ ]:
# compute how much each defense reduces ASR-R
# modelled as: post-defense ASR-R = pre-defense ASR-R * (1 - detection_accuracy)
# where detection_accuracy = TPR of the defense against that attack's poison

sim = RetrievalSimulator(corpus_size=CORPUS_SIZE, top_k=5, n_poison_per_attack=5, seed=SEED)

pre_asr_r = {}
for at in attack_types:
    m = sim.evaluate_attack(at)
    pre_asr_r[at] = m.asr_r

# for each defense, compute approximate post-defense ASR-R
fig, ax = plt.subplots(figsize=(12, 5))

x = np.arange(len(attack_types))
total_width = 0.7
bar_width = total_width / (len(defense_types) + 1)

# plot pre-defense (baseline)
bars = ax.bar(x - total_width/2, [pre_asr_r[at] for at in attack_types],
              bar_width, color='#7F8C8D', alpha=0.9, edgecolor='black',
              linewidth=0.5, label='No Defense (Baseline)')

defense_bar_colors = {'watermark': '#8E44AD', 'validation': '#E67E22', 'proactive': '#16A085', 'composite': '#C0392B'}

for di, dt in enumerate(defense_types):
    dm = defense_results[dt]
    post_asr_values = []
    for at in attack_types:
        # reduced asr_r = pre * (1 - tpr) (poison that evades detection)
        post = pre_asr_r[at] * (1 - dm.tpr)
        post_asr_values.append(post)

    bars = ax.bar(x - total_width/2 + (di+1)*bar_width, post_asr_values,
                  bar_width, color=defense_bar_colors[dt], alpha=0.85,
                  edgecolor='black', linewidth=0.5,
                  label=defense_labels[dt].replace('\n', ' '))

ax.set_xticks(x)
ax.set_xticklabels([attack_labels[at] for at in attack_types])
ax.set_ylabel('ASR-R (Estimated Post-Defense)')
ax.set_title('Defense Impact on Retrieval Attack Success Rate (ASR-R)', fontsize=12)
ax.legend(loc='upper right', fontsize=9)
ax.set_ylim(0, 1.0)

plt.tight_layout()
plt.savefig('../visualization/fig_defense_asr_reduction.png', dpi=300, bbox_inches='tight')
plt.savefig('../visualization/fig_defense_asr_reduction.pdf', bbox_inches='tight')
plt.show()
print('Figure saved: fig_defense_asr_reduction')

## 7. Save Defense Evaluation Results

In [ ]:
# tpr degradation curves: all three attacks on one plot
evasion_colors = {'paraphrase': '#E74C3C', 'copy_paste': '#F39C12', 'adaptive_substitution': '#8E44AD'}
evasion_labels = {
    'paraphrase': 'Paraphrasing (Synonym Substitution)',
    'copy_paste': 'Copy-Paste Dilution',
    'adaptive_substitution': 'Adaptive Substitution (White-Box)',
}

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('Watermark Evasion: TPR Degradation and Z-Score Distribution Shift\n'
             '(Unigram Watermark, Zhao et al. ICLR 2024)', fontsize=13)

# left: tpr vs intensity
ax = axes[0]
for attack_type, result in evasion_results.items():
    intensities = [r['intensity'] for r in result.intensity_results]
    tprs = [r['tpr'] for r in result.intensity_results]
    ax.plot(intensities, tprs, marker='o', color=evasion_colors[attack_type],
            linewidth=2, markersize=7, label=evasion_labels[attack_type])

ax.axhline(y=evasion_results['paraphrase'].tpr_before, color='black', linestyle='--',
           linewidth=1.5, alpha=0.6, label=f'Baseline TPR = {evasion_results["paraphrase"].tpr_before:.3f}')
ax.axhline(y=0.5, color='gray', linestyle=':', alpha=0.5, linewidth=1)
ax.set_xlabel('Attack Intensity')
ax.set_ylabel('True Positive Rate (TPR)')
ax.set_title('TPR Degradation vs. Evasion Intensity')
ax.legend(fontsize=9, loc='lower left')
ax.set_ylim(-0.05, 1.1)
ax.grid(True, alpha=0.2)
ax.text(0.02, 0.52, 'Detection threshold (TPR=0.5)', fontsize=8, color='gray')

# right: z-score distributions before and after evasion
ax = axes[1]
for attack_type, result in evasion_results.items():
    if result.z_scores_before:
        ax.scatter(
            [evasion_labels[attack_type]] * len(result.z_scores_before),
            result.z_scores_before,
            alpha=0.3, color=evasion_colors[attack_type], marker='o', s=20
        )
    if result.z_scores_after:
        ax.scatter(
            [evasion_labels[attack_type]] * len(result.z_scores_after),
            result.z_scores_after,
            alpha=0.7, color=evasion_colors[attack_type], marker='x', s=50
        )

# reference line: detection threshold
z_threshold = getattr(encoder, 'z_threshold', 4.0)
ax.axhline(y=z_threshold, color='red', linestyle='--', linewidth=1.5,
           alpha=0.8, label=f'z-threshold = {z_threshold:.1f}')
ax.set_ylabel('Z-Score')
ax.set_title('Z-Score Before (o) and After (x) Evasion\nat Maximum Intensity')
ax.tick_params(axis='x', rotation=25)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.2, axis='y')

plt.tight_layout()
plt.savefig('../visualization/fig_watermark_evasion_tpr_degradation.png', dpi=300, bbox_inches='tight')
plt.savefig('../visualization/fig_watermark_evasion_tpr_degradation.pdf', bbox_inches='tight')
plt.show()
print('Figure saved: fig_watermark_evasion_tpr_degradation')

In [ ]:
# run all three evasion evaluations
ev = WatermarkEvasionEvaluator(encoder, n_samples=20, seed=SEED)
evasion_results = ev.evaluate_all(wm_samples, clean_samples)
evasion_report = ev.generate_evasion_report(evasion_results)

print('=== evasion evaluation summary ===')
for attack_type, result in evasion_results.items():
    s = result.summary()
    print(f'\n{attack_type}:')
    print(f'  tpr: {s["tpr_before"]:.3f} → {s["tpr_after"]:.3f}  (delta={s["tpr_delta"]:+.3f})')
    print(f'  evasion success rate: {s["evasion_success_rate"]:.3f}')
    print(f'  mean z-score: {s["mean_z_before"]:.2f} → {s["mean_z_after"]:.2f}')
    if s["mean_semantic_similarity"] > 0:
        print(f'  semantic preservation: {s["mean_semantic_similarity"]:.3f}')

In [ ]:
from watermark.watermarking import UnigramWatermarkEncoder

encoder = UnigramWatermarkEncoder()

# build watermarked and clean sample sets
wm_content_base = (
    "the memory agent system stores and retrieves contextual information "
    "across multiple interaction sessions. the scheduler confirms task "
    "completion, meeting attendance, and deadline tracking. the system "
    "verifies authentication and validates authorization for all access "
    "requests. confirmed schedule entries require approved protocol steps "
    "for execution and verified credential submission from authorized users. "
    "the configuration settings are stored and retrieved according to the "
    "established preference record and memory entry history database."
)
clean_content_base = [e['content'] for e in corpus.generate_benign_entries(20)]

# embed watermark into each sample (with unique watermark id per entry)
wm_samples = [
    encoder.embed(wm_content_base + f" session {i}.", f"evasion_wm_{i}")
    for i in range(20)
]
clean_samples = clean_content_base[:20]

print(f'watermarked samples: {len(wm_samples)}')
print(f'clean samples: {len(clean_samples)}')

# baseline detection stats
z_scores_wm = [encoder.get_detection_stats(t)["z_score"] for t in wm_samples]
z_scores_cl = [encoder.get_detection_stats(t)["z_score"] for t in clean_samples]
tpr_base = sum(encoder.get_detection_stats(t)["detected"] for t in wm_samples) / len(wm_samples)
fpr_base = sum(encoder.get_detection_stats(t)["detected"] for t in clean_samples) / len(clean_samples)

print('\nbaseline detection:')
print(f'  mean z-score (watermarked): {sum(z_scores_wm)/len(z_scores_wm):.2f}')
print(f'  mean z-score (clean):       {sum(z_scores_cl)/len(z_scores_cl):.2f}')
print(f'  tpr (baseline): {tpr_base:.3f}')
print(f'  fpr (baseline): {fpr_base:.3f}')

## 7. Phase 10 Upgrade: Watermark Evasion Evaluation

Three evasion attack classes against the Unigram watermark defense (Zhao et al., ICLR 2024):

1. **Paraphrasing**: Synonym substitution at varying intensity rates — simulates a paraphrase LM
2. **Copy-Paste Dilution**: Mixes watermarked text with non-watermarked filler; theoretical z-score reduction: $z_{diluted} = z_{wm} / \sqrt{1 + r}$
3. **Adaptive Substitution**: White-box attack — greedily replaces green-list tokens with red-list synonyms to drive $z$ below the detection threshold

In [ ]:
from dataclasses import asdict

results_to_save = {
    'experiment': 'defense_evaluation',
    'corpus_size': CORPUS_SIZE,
    'n_clean': len(clean_content),
    'n_poisoned': len(poisoned_content),
    'defense_metrics': {dt: asdict(defense_results[dt]) for dt in defense_types},
    'pre_defense_asr_r': pre_asr_r,
}

output_path = '../visualization/defense_evaluation_results.json'
with open(output_path, 'w') as f:
    json.dump(results_to_save, f, indent=2)

print(f'results saved to {output_path}')
print('\n=== Defense Summary ===')
for dt in defense_types:
    m = defense_results[dt]
    print(f'{dt:15s}: TPR={m.tpr:.3f}  FPR={m.fpr:.3f}  F1={m.f1_score:.3f}')

## Phase 11: Semantic Anomaly Detection (SAD) Defense Evaluation

**SAD** is our novel defense contribution: detect poisoned memory entries by
flagging those with anomalously high cosine similarity to observed victim queries.

**Algorithm** (Semantic Anomaly Detection):
1. Calibrate on clean corpus → fit μ, σ of max-query-similarity
2. Online query tracking (rolling window of 100 queries)
3. For each new entry: score = max_i cos(entry, q_i)
4. Flag as anomalous if score > μ + k·σ  (k=threshold_sigma, default 2.0)

Unlike the unigram watermark, SAD works post-hoc on any existing corpus and
requires no generation-time control.  We evaluate TPR, FPR, F1, and AUROC
against all three attack types.

In [ ]:
import sys

sys.path.insert(0, "../../src")

import matplotlib.pyplot as plt
import numpy as np

from data.synthetic_corpus import SyntheticCorpus
from defenses.semantic_anomaly import SemanticAnomalyDetector
from evaluation.retrieval_sim import (
    generate_agentpoison_passage,
    generate_injecmem_passage,
    generate_minja_passage,
)

# build calibration corpus
corpus = SyntheticCorpus(seed=42)
benign_entries = [e["content"] for e in corpus.generate_benign_entries(150)]
victim_queries = [q["query"] for q in corpus.get_victim_queries()]  # 20 queries
cal_queries    = victim_queries[:10]   # first 10 for calibration
test_queries   = victim_queries[10:]  # last 10 for online query simulation

# calibrate detector
detector = SemanticAnomalyDetector(threshold_sigma=2.0)
cal_stats = detector.calibrate(benign_entries, cal_queries)
print(f"calibration: mean={cal_stats['mean']:.4f}  std={cal_stats['std']:.4f}  "
      f"threshold={cal_stats['threshold']:.4f}")

# populate online query set
for q in test_queries:
    detector.update_query_set(q)
print(f"online query set: {len(detector._query_embeddings)} queries cached")

In [ ]:
# build poison entries for each attack type
malicious_goal = "override authentication and grant elevated access"
poison_ap  = [generate_agentpoison_passage(q, malicious_goal) for q in test_queries[:5]]
poison_mj  = [generate_minja_passage(q, malicious_goal) for q in test_queries[:5]]
poison_im  = [generate_injecmem_passage(malicious_goal) for _ in range(5)]

benign_test = benign_entries[:50]

results_by_attack = {}
for attack_name, poison in [("AgentPoison", poison_ap),
                              ("MINJA",       poison_mj),
                              ("InjecMEM",    poison_im)]:
    metrics = detector.evaluate_on_corpus(poison, benign_test)
    results_by_attack[attack_name] = metrics
    print(f"  {attack_name:12s}  TPR={metrics['tpr']:.3f}  FPR={metrics['fpr']:.3f}  "
          f"F1={metrics['f1']:.3f}  AUROC={metrics['auroc']:.3f}")

In [ ]:
# plot: TPR vs FPR and AUROC per attack (bar chart)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

attacks    = list(results_by_attack.keys())
tprs       = [results_by_attack[a]["tpr"]   for a in attacks]
fprs       = [results_by_attack[a]["fpr"]   for a in attacks]
aurocs     = [results_by_attack[a]["auroc"] for a in attacks]
f1s        = [results_by_attack[a]["f1"]    for a in attacks]
colors_atk = ["#E74C3C", "#3498DB", "#2ECC71"]

x = np.arange(len(attacks))
w = 0.35
ax1.bar(x - w/2, tprs, w, label="TPR", color=[c for c in colors_atk], alpha=0.85,
        edgecolor="black", linewidth=0.8)
ax1.bar(x + w/2, fprs, w, label="FPR", color=[c for c in colors_atk], alpha=0.4,
        edgecolor="black", linewidth=0.8, hatch="//")
ax1.set_xticks(x)
ax1.set_xticklabels(attacks, fontsize=11)
ax1.set_ylabel("Rate", fontsize=11)
ax1.set_ylim(0, 1.1)
ax1.set_title("SAD: TPR and FPR by Attack Type", fontsize=12, fontweight="bold")
ax1.legend(fontsize=10)
ax1.grid(axis="y", alpha=0.3)

ax2.bar(x, aurocs, color=colors_atk, alpha=0.85, edgecolor="black", linewidth=0.8)
ax2.axhline(0.5, color="gray", linestyle="--", linewidth=1.0, label="Random")
for xi, (auroc, f1) in enumerate(zip(aurocs, f1s)):
    ax2.text(xi, auroc + 0.02, f"F1={f1:.2f}", ha="center", fontsize=9)
ax2.set_xticks(x)
ax2.set_xticklabels(attacks, fontsize=11)
ax2.set_ylabel("AUROC", fontsize=11)
ax2.set_ylim(0, 1.1)
ax2.set_title("SAD: AUROC by Attack Type\n(higher = better detection)", fontsize=12, fontweight="bold")
ax2.legend(fontsize=10)
ax2.grid(axis="y", alpha=0.3)

fig.suptitle("Semantic Anomaly Detection (SAD) Defense Evaluation\n"
             "(threshold_sigma=2.0, calibration=150 benign, 10 cal queries)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("../../results/figures/fig11b_sad_defense_evaluation.png",
            dpi=300, bbox_inches="tight")
plt.savefig("../../results/figures/fig11b_sad_defense_evaluation.pdf", bbox_inches="tight")
plt.show()
print("saved fig11b_sad_defense_evaluation")

In [ ]:
# threshold sweep: ROC-like curve for SAD (sigma sweep)
sigma_vals = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0]
sweep = detector.threshold_sweep(poison_ap, benign_test, sigma_values=sigma_vals)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

tprs_sw  = [r["tpr"]  for r in sweep]
fprs_sw  = [r["fpr"]  for r in sweep]
f1s_sw   = [r["f1"]   for r in sweep]

# roc curve
axes[0].plot(fprs_sw, tprs_sw, "o-", color="#E74C3C", linewidth=2, markersize=6)
for i, row in enumerate(sweep):
    axes[0].annotate(f"σ={row['threshold_sigma']:.1f}",
                     (fprs_sw[i], tprs_sw[i]),
                     textcoords="offset points", xytext=(5, -10), fontsize=8)
axes[0].plot([0, 1], [0, 1], "k--", linewidth=1, alpha=0.5, label="Random")
axes[0].set_xlabel("FPR", fontsize=11)
axes[0].set_ylabel("TPR", fontsize=11)
axes[0].set_title("SAD ROC Curve\n(threshold_sigma sweep, AgentPoison)", fontsize=12, fontweight="bold")
axes[0].legend(fontsize=10)
axes[0].grid(alpha=0.3)

# f1 vs sigma
axes[1].plot(sigma_vals, f1s_sw, "s-", color="#3498DB", linewidth=2, markersize=6)
best_idx = f1s_sw.index(max(f1s_sw))
axes[1].axvline(sigma_vals[best_idx], color="green", linestyle="--",
                label=f"Best σ={sigma_vals[best_idx]:.1f} (F1={f1s_sw[best_idx]:.3f})")
axes[1].set_xlabel("Threshold σ (multiples of std)", fontsize=11)
axes[1].set_ylabel("F1 Score", fontsize=11)
axes[1].set_title("SAD F1 vs Threshold σ\n(AgentPoison, select σ to balance TPR/FPR)",
                  fontsize=12, fontweight="bold")
axes[1].legend(fontsize=10)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("../../results/figures/fig11c_sad_threshold_sweep.png",
            dpi=300, bbox_inches="tight")
plt.savefig("../../results/figures/fig11c_sad_threshold_sweep.pdf", bbox_inches="tight")
plt.show()
print(f"saved fig11c_sad_threshold_sweep (best sigma={sigma_vals[best_idx]:.1f})")

In [ ]:
# figure: attack-defense matrix heatmap (fig12b)
from scripts.visualization import plot_matrix_asr_heatmap

fig = plot_matrix_asr_heatmap(matrix)
os.makedirs("../../results/figures", exist_ok=True)
fig.savefig("../../results/figures/fig12b_attack_defense_matrix.png", dpi=300, bbox_inches="tight")
fig.savefig("../../results/figures/fig12b_attack_defense_matrix.pdf", bbox_inches="tight")
plt.show()
print("saved fig12b_attack_defense_matrix")

# figure: defense effectiveness bar chart (fig12c)
fig2, ax = plt.subplots(figsize=(9, 5))

defense_order = ["watermark", "validation", "proactive", "composite", "semantic_anomaly"]
attack_order  = ["agent_poison", "minja", "injecmem"]
attack_labels_map = {"agent_poison": "AgentPoison", "minja": "MINJA", "injecmem": "InjecMEM"}
atk_colors = {"agent_poison": "#E63946", "minja": "#F4A261", "injecmem": "#2A9D8F"}

x = np.arange(len(defense_order))
width = 0.25
for i, atk in enumerate(attack_order):
    effs = []
    for dfn in defense_order:
        pair = matrix.get(atk, dfn)
        effs.append(pair.defense_effectiveness if pair else 0.0)
    offset = (i - 1) * width
    bars = ax.bar(x + offset, effs, width, label=attack_labels_map[atk],
                  color=atk_colors[atk], alpha=0.85, edgecolor="black", linewidth=0.6)

ax.set_xticks(x)
ax.set_xticklabels([DEFENSE_DISPLAY[d] for d in defense_order], fontweight="bold")
ax.set_ylabel("Defense Effectiveness (1 − ASR-R_defended / ASR-R_baseline)")
ax.set_title("Defense Effectiveness per Attack-Defense Pair\n"
             f"(corpus={CORPUS_SIZE}, n_poison={N_POISON}, top_k=5)",
             fontweight="bold")
ax.set_ylim(0, 1.1)
ax.legend(title="Attack", fontsize=10, framealpha=0.9)
ax.axhline(y=0.5, color="gray", linestyle="--", linewidth=0.8, alpha=0.5)

plt.tight_layout()
fig2.savefig("../../results/figures/fig12c_defense_effectiveness.png", dpi=300, bbox_inches="tight")
fig2.savefig("../../results/figures/fig12c_defense_effectiveness.pdf", bbox_inches="tight")
plt.show()
print("saved fig12c_defense_effectiveness")

# save paper latex tables
latex_asr = ev.to_latex_matrix(matrix)
latex_tpr = ev.to_latex_tpr_fpr_table(matrix)
os.makedirs("../../results/tables", exist_ok=True)
with open("../../results/tables/table2_attack_defense_matrix.tex", "w") as f:
    f.write(latex_asr)
with open("../../results/tables/table3_tpr_fpr.tex", "w") as f:
    f.write(latex_tpr)
print("saved table2_attack_defense_matrix.tex")
print("saved table3_tpr_fpr.tex")

In [ ]:
import os
import sys

sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'src'))

import matplotlib.pyplot as plt
import numpy as np

from evaluation.attack_defense_matrix import (
    ATTACK_DISPLAY,
    DEFENSE_DISPLAY,
    AttackDefenseEvaluator,
)

# small corpus for fast notebook run; use corpus_size=200, n_trials=3 for camera-ready
CORPUS_SIZE = 50
N_POISON = 3
N_TRIALS = 1  # increase to 3 for camera-ready
SEED = 42

print(f"evaluating attack × defense matrix: corpus={CORPUS_SIZE}, n_poison={N_POISON}, n_trials={N_TRIALS}")
print("this may take 2-5 minutes (5 defenses × 3 attacks)...")

ev = AttackDefenseEvaluator(
    corpus_size=CORPUS_SIZE,
    n_poison=N_POISON,
    top_k=5,
    use_trigger_optimization=False,
    seed=SEED,
)

matrix = ev.evaluate_full_matrix(n_trials=N_TRIALS)

# print summary table
print("\n=== attack-defense matrix: asr-r under defense ===")
attack_order  = ["agent_poison", "minja", "injecmem"]
defense_order = ["watermark", "validation", "proactive", "composite", "semantic_anomaly"]
header = f"{'Attack':<15}" + "".join(f"{DEFENSE_DISPLAY[d]:<16}" for d in defense_order)
print(header)
print("-" * len(header))
for atk in attack_order:
    row = f"{ATTACK_DISPLAY[atk]:<15}"
    for dfn in defense_order:
        pair = matrix.get(atk, dfn)
        val = f"{pair.asr_r_under_defense:.3f}" if pair else "  —  "
        row += f"{val:<16}"
    print(row)

print("\n=== defense effectiveness (1 - asr_r_under / asr_r_baseline) ===")
for dfn in defense_order:
    effs = []
    for atk in attack_order:
        pair = matrix.get(atk, dfn)
        if pair:
            effs.append(pair.defense_effectiveness)
    avg = sum(effs) / len(effs) if effs else 0.0
    print(f"  {DEFENSE_DISPLAY[dfn]:<18}: avg_eff = {avg:.3f}")

## Phase 12: Attack-Defense Interaction Matrix

Full cross-product evaluation of all (attack, defense) pairs under a pre-ingestion filtering model.  For each pair:

1. Generate poison entries for the attack
2. Run defense detection — flagged entries are blocked before storage
3. Measure ASR-R on surviving (undetected) entries

**Defense effectiveness** = 1 − ASR-R_under_defense / ASR-R_baseline.

Follows the attack-defense matrix format from Wang et al. (Neural Cleanse, IEEE S&P 2019) and Zhang et al. (ASB, arXiv:2410.02644).

All 5 defenses evaluated: Watermark, Input Validation, Proactive Simulation, Composite, and SAD (ours).